# 02 — Preprocesado y particiones

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MiltonCuervo/ham10000-skin-lesion-classification/blob/main/02_preprocesado.ipynb)

Ejecución local: abre Jupyter desde la carpeta del proyecto y selecciona **Python (HAM10000 local GPU)**. Ejecuta en orden 01–05. Los resultados se guardan localmente; no se monta Drive. En Colab, usa `/content` como carpeta del proyecto y ejecuta primero 01 y 02.


## 1. Configuración

In [1]:
from pathlib import Path
import os, json, hashlib, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
PROJECT = Path.cwd().resolve()  # Abrir Jupyter desde la carpeta del proyecto.
DATA = PROJECT/'data'/'ham10000'
SPLITS = PROJECT/'data'/'splits'
CACHE = PROJECT/'data'/'processed_224'
SEED = 42
CLASSES = ['akiec','bcc','bkl','df','mel','nv','vasc']
POSITIVE = {'mel','bcc','akiec'}

In [2]:
def index_images(root):
    index = {}
    for path in sorted(root.rglob('*.jpg')):
        if path.stem in index:
            other = index[path.stem]
            if hashlib.sha256(other.read_bytes()).digest() != hashlib.sha256(path.read_bytes()).digest():
                raise ValueError(f'Contenido diferente para {path.stem}: {other} / {path}')
        else:
            index[path.stem] = path
    return index

## 2. Particiones por lesión

Se conservan los manifiestos existentes. Si no existen, se usa 70/15/15 aproximado con semilla fija. Nunca se mueve una lesión entre particiones para mejorar métricas.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
meta=pd.read_csv(DATA/'HAM10000_metadata.csv')
assert len(meta)==10015 and meta.image_id.is_unique
assert not meta[['image_id','lesion_id','dx']].isna().any().any()
assert meta.groupby('lesion_id').dx.nunique().max()==1
SPLITS.mkdir(parents=True,exist_ok=True)
files={name:SPLITS/f'{name}.csv' for name in ['train','valid','test']}
if any(p.exists() for p in files.values()) and not all(p.exists() for p in files.values()):
    raise RuntimeError('Faltan manifiestos: restaura las tres particiones antes de continuar.')
if all(p.exists() for p in files.values()):
    parts={name:pd.read_csv(path)[['image_id','lesion_id','dx']] for name,path in files.items()}
else:
    a,b=next(GroupShuffleSplit(n_splits=1,test_size=.30,random_state=SEED).split(meta,groups=meta.lesion_id))
    rest=meta.iloc[b]
    c,d=next(GroupShuffleSplit(n_splits=1,test_size=.50,random_state=SEED).split(rest,groups=rest.lesion_id))
    parts={'train':meta.iloc[a],'valid':rest.iloc[c],'test':rest.iloc[d]}
    parts={name:frame[['image_id','lesion_id','dx']].copy() for name,frame in parts.items()}
combined=pd.concat(parts.values()).sort_values('image_id').reset_index(drop=True)
expected=meta[['image_id','lesion_id','dx']].sort_values('image_id').reset_index(drop=True)
pd.testing.assert_frame_equal(combined,expected)
for a,b in [('train','valid'),('train','test'),('valid','test')]:
    assert set(parts[a].lesion_id).isdisjoint(parts[b].lesion_id)
    assert set(parts[a].image_id).isdisjoint(parts[b].image_id)
for name,frame in parts.items():
    assert frame.image_id.is_unique and set(frame.dx)==set(CLASSES)
    frame.to_csv(files[name],index=False)
display(pd.DataFrame({name:{'imágenes':len(f),'lesiones':f.lesion_id.nunique()} for name,f in parts.items()}).T)
display(pd.DataFrame({name:f.dx.value_counts() for name,f in parts.items()}).fillna(0))

,imágenes,lesiones
train,7002,5229
valid,1519,1120
test,1494,1121


,train,valid,test
dx,,,
akiec,207,57,63
bcc,381,65,68
bkl,741,206,152
df,89,19,7
mel,770,156,187
nv,4718,991,996
vasc,96,25,21


## 3. Integridad y caché local 224×224

Se verifica cada original y se detectan duplicados exactos entre particiones. La caché guarda el resultado del redimensionado bilineal RGB en PNG sin pérdida adicional; evita decodificar y redimensionar los JPG originales en cada época. Aumento y normalización se aplican después, durante entrenamiento. No se alteran los originales.

In [4]:
CACHE.mkdir(parents=True,exist_ok=True)
paths=index_images(DATA); records=[]
previous=SPLITS/'image_audit.csv'
previous_hashes=pd.read_csv(previous).set_index('image_id').sha256.to_dict() if previous.exists() else {}
for split,frame in parts.items():
    for row in tqdm(frame.itertuples(),total=len(frame),desc=split):
        source=paths[row.image_id]
        with Image.open(source) as image: image.verify()
        sha=hashlib.sha256(source.read_bytes()).hexdigest()
        records.append({'image_id':row.image_id,'lesion_id':row.lesion_id,'split':split,'sha256':sha})
        target=CACHE/f'{row.image_id}.png'
        cache_valid=False
        if target.exists() and previous_hashes.get(row.image_id)==sha:
            try:
                with Image.open(target) as cached:
                    cache_valid=cached.mode=='RGB' and cached.size==(224,224); cached.verify()
            except Exception: pass
        if not cache_valid:
            with Image.open(source) as image:
                resized=image.convert('RGB').resize((224,224),Image.Resampling.BILINEAR)
                temporary=target.with_suffix('.tmp'); resized.save(temporary,format='PNG',compress_level=1)
                os.replace(temporary,target)
audit=pd.DataFrame(records)
cross=audit.groupby('sha256')['split'].nunique()
assert not (cross>1).any(), 'Duplicados exactos entre particiones: revisar antes de entrenar.'
audit.to_csv(SPLITS/'image_audit.csv',index=False)
manifest={'split_hash':hashlib.sha256(json.dumps({n:f.to_dict('records') for n,f in parts.items()},sort_keys=True).encode()).hexdigest(),
          'data_hash':hashlib.sha256(audit[['image_id','sha256']].sort_values('image_id').to_csv(index=False).encode()).hexdigest(),
          'cache':'RGB_224_bilinear_PNG_v1','images':len(audit)}
(SPLITS/'manifest_local.json').write_text(json.dumps(manifest,indent=2))
print('Preparación completada:',manifest)

train:   0%|          | 0/7002 [00:00<?, ?it/s]

valid:   0%|          | 0/1519 [00:00<?, ?it/s]

test:   0%|          | 0/1494 [00:00<?, ?it/s]

Preparación completada: {'split_hash': '07c4ac8729b993e278f21e19e8843877e53d261d8818502152d637d2fc23895a', 'data_hash': 'da62087684a7371bfe20cfe3980c4a6ce122c081a67297f2ab12c69ffcf84ed7', 'cache': 'RGB_224_bilinear_PNG_v1', 'images': 10015}
